In [2]:
import torch
from clean_lib.checkpoints import CheckpointManager
from clean_lib.sae import SparseAEs, Normalizer
from clean_lib.processors.processor import Processor
from clean_lib.processors.H import H
from clean_lib.processors.D import D
from clean_lib.processors.R import R
from clean_lib.processors.Hc import Hc
from clean_lib.processors.M import M
from clean_lib.analyzer import Analyzer
from clean_lib.visualize import visualize_concept_on_class

In [7]:
N = 3
backbone_dir = "./PACS_ResNet_Sketch_Test_Only/ERM_ResNet_T3"

backbone_manager = CheckpointManager(directory=backbone_dir)
ckpts_oracle, _ = backbone_manager.get_top_k_checkpoints(envs=[0, 1, 2, 3], k = N)
ckpts_nonoracle, _ = backbone_manager.get_top_k_checkpoints(envs=[0, 1, 2], k = N)

print(f"Top - {N} checkpoints Oracle: {ckpts_oracle}")
print(f"Top - {N} checkpoints Non - Oracle: {ckpts_nonoracle}")

backbone_manager.load_checkpoints(set(ckpts_oracle).union(set(ckpts_nonoracle)))

Top - 3 checkpoints Oracle: [2100, 4200, 3300]
Top - 3 checkpoints Non - Oracle: [3300, 2100, 1500]


In [ ]:
sae_manager = SparseAEs(
    feature_dim=2048, 
    sae_dim=128, 
    topk=16, 
    nb_concepts=2048*8, 
    rearrange_string="n c w h -> (n w h) c", 
    checkpointManager=backbone_manager, 
    train_envs=[0, 1, 2],
    w=7
)

sae_manager.configure_training()
sae_manager.train(flag="USAE_3300_2100_1500_4200", epochs=500, batch_size=64, full_data_gpu=True, save_dir="./SAEs", dataset="PACS")

Training: USAE_3300_2100_1500_4200_ERM_ResNet_T3:  44%|████▎     | 218/500 [1:01:53<1:34:06, 20.02s/it, loss=2.100165]